# Retail Orders — Data Ingestion, Cleaning & Preprocessing

**Objective:** Clean the supplied raw retail-orders dataset using Pandas, document quality issues, perform feature engineering, and export a standardized `clean_dataset.csv`.

> **Important data-volume note:** The supplied `retail-orders-raw.csv` contains **12 rows**, not 10,000+. The notebook therefore cleans the supplied data as-is and does **not** fabricate additional records. The same pipeline can be applied unchanged to a 10,000+ row source file.

**Data dictionary:** `retail-data-dictionary.csv` was used to validate required fields and business rules.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path("retail-orders-raw.csv")
DICT_FILE = Path("retail-data-dictionary.csv")
OUTPUT_FILE = Path("clean_dataset.csv")

raw = pd.read_csv(RAW_FILE)
data_dictionary = pd.read_csv(DICT_FILE)

print(f"Raw shape: {raw.shape}")
display(raw.head(10))
display(data_dictionary)

Raw shape: (12, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid


,column,data_type,business_definition,quality_rule
0,order_id,string,Unique order identifier,Required and unique
1,order_date,date,Date the customer placed the order,Required ISO date between 2025-01-01 and today
2,customer_segment,category,Commercial customer segment,Student Fresher or Professional after normaliz...
3,city,string,Customer billing city,Required non-empty text
4,category,category,Product family,Learning Kit Course Access or Mentor Session
5,quantity,integer,Units purchased,Whole number greater than zero
6,unit_price,decimal,Price per unit before discount,Non-negative INR amount
7,discount_pct,decimal,Percentage discount applied,Between 0 and 100; missing means zero only aft...
8,payment_status,category,Latest order settlement state,Paid Pending Failed or Refunded after normaliz...


## 1. Before cleaning — data quality profile

In [2]:
print("Rows and columns:", raw.shape)
print("\nData types:")
display(raw.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(raw.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", raw.duplicated().sum())

print("\nUnique values in key categorical fields:")
for col in ["customer_segment", "city", "category", "payment_status", "quantity"]:
    print(f"\n{col}:")
    print(raw[col].value_counts(dropna=False).to_string())

Rows and columns: (12, 9)

Data types:


,dtype
order_id,object
order_date,object
customer_segment,object
city,object
category,object
quantity,object
unit_price,int64
discount_pct,float64
payment_status,object



Missing values:


,missing_count
order_id,0
order_date,1
customer_segment,0
city,1
category,0
quantity,0
unit_price,0
discount_pct,1
payment_status,0


Exact duplicate rows: 1

Unique values in key categorical fields:

customer_segment:
customer_segment
Student         4
Professional    4
Fresher         3
student         1

city:
city
Chennai      3
Bengaluru    2
Hyderabad    2
NaN          1
Pune         1
Mumbai       1
Delhi        1
Kochi        1

category:
category
Learning Kit      5
Course Access     4
Mentor Session    3

payment_status:
payment_status
Paid        7
Pending     2
paid        1
Failed      1
Refunded    1

quantity:
quantity
1      5
2      3
3      2
-1     1
two    1


In [3]:
display(raw)

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,NaN,Mentor Session,1,999,0.0,Failed
6,RT-1006,2026-13-10,Student,Pune,Learning Kit,-1,799,10.0,Paid
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,105.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,two,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid


### Issues found

- `order_date` mixes ISO and slash-formatted dates and contains a missing value plus an invalid date (`2026-13-10`).
- `quantity` is stored as text and includes a word value (`two`) and an invalid negative quantity (`-1`).
- `discount_pct` has a missing value and an invalid value above the allowed 0–100% range (`105`).
- One exact duplicate order record is present.
- Text categories have inconsistent capitalization (`Student` vs `student`, `Paid` vs `paid`).
- `city` has a missing value.
- The data dictionary requires `order_id` to be unique, dates to be valid, quantity to be positive, and categorical fields to use standardized values.

## 2. Cleaning and standardization

In [4]:
clean = raw.copy()

# Standardize text fields
for col in ["customer_segment", "city", "category", "payment_status"]:
    clean[col] = clean[col].astype("string").str.strip()

clean["customer_segment"] = clean["customer_segment"].str.title()
clean["city"] = clean["city"].str.title()
clean["category"] = clean["category"].str.replace(r"\s+", " ", regex=True).str.title()
clean["payment_status"] = clean["payment_status"].str.title()

# Parse dates; invalid/missing dates become NaT
clean["order_date"] = pd.to_datetime(clean["order_date"], errors="coerce")

# Convert quantity, including common word-form values
quantity_words = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5"}
clean["quantity"] = (
    clean["quantity"].astype("string").str.strip().str.lower().replace(quantity_words)
)
clean["quantity"] = pd.to_numeric(clean["quantity"], errors="coerce")

# Convert numeric fields
clean["unit_price"] = pd.to_numeric(clean["unit_price"], errors="coerce")
clean["discount_pct"] = pd.to_numeric(clean["discount_pct"], errors="coerce")

# Remove exact duplicate records
clean = clean.drop_duplicates().copy()

# Required transaction date cannot be reliably inferred, so discard rows with
# missing/invalid dates rather than inventing a transaction date.
clean = clean.loc[clean["order_date"].notna()].copy()

# Quantity must be a whole number greater than zero.
clean = clean.loc[clean["quantity"].notna() & (clean["quantity"] > 0)].copy()
clean["quantity"] = clean["quantity"].astype("int64")

# Discount must be 0–100. Invalid values are treated as missing and then
# imputed as 0, consistent with the data dictionary's "missing means zero"
# rule for this field.
invalid_discount = (clean["discount_pct"] < 0) | (clean["discount_pct"] > 100)
clean.loc[invalid_discount, "discount_pct"] = np.nan
clean["discount_pct"] = clean["discount_pct"].fillna(0)

# Required city: retain the transaction but make missingness explicit.
clean["city"] = clean["city"].fillna("Unknown")

# Validate categorical domains from the data dictionary.
valid_values = {
    "customer_segment": {"Student", "Fresher", "Professional"},
    "category": {"Learning Kit", "Course Access", "Mentor Session"},
    "payment_status": {"Paid", "Pending", "Failed", "Refunded"},
}
for col, allowed in valid_values.items():
    clean.loc[~clean[col].isin(allowed), col] = "Unknown"

# Standardized date format
clean["order_date"] = clean["order_date"].dt.normalize()

print("Shape after cleaning rules:", clean.shape)
display(clean)

Shape after cleaning rules: (8, 9)


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1,1499,0.0,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
5,RT-1005,2026-01-09,Fresher,Unknown,Mentor Session,1,999,0.0,Failed
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,0.0,Paid
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,2,1499,0.0,Pending
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid
10,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15.0,Refunded


## 3. Outlier and business-rule checks

In [5]:
# IQR diagnostics for numeric variables.
# With a very small supplied dataset, IQR results are treated as diagnostics,
# while explicit business rules take precedence.
def iqr_outliers(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lower, upper, series[(series < lower) | (series > upper)]

for col in ["quantity", "unit_price", "discount_pct"]:
    lower, upper, flagged = iqr_outliers(clean[col])
    print(f"{col}: IQR bounds=({lower:.2f}, {upper:.2f}); flagged rows={len(flagged)}")

print("\nBusiness-rule checks:")
print("Duplicate order IDs:", clean["order_id"].duplicated().sum())
print("Invalid/non-positive quantities:", (clean["quantity"] <= 0).sum())
print("Invalid discounts:", ((clean["discount_pct"] < 0) | (clean["discount_pct"] > 100)).sum())
print("Missing dates:", clean["order_date"].isna().sum())
print("Missing cities:", clean["city"].isna().sum())

quantity: IQR bounds=(-0.50, 3.50); flagged rows=0
unit_price: IQR bounds=(-251.00, 2549.00); flagged rows=0
discount_pct: IQR bounds=(-9.38, 15.62); flagged rows=0

Business-rule checks:
Duplicate order IDs: 0
Invalid/non-positive quantities: 0
Invalid discounts: 0
Missing dates: 0
Missing cities: 0


## 4. Feature engineering

In [6]:
# Revenue-related features
clean["gross_sales_inr"] = clean["quantity"] * clean["unit_price"]
clean["discount_amount_inr"] = clean["gross_sales_inr"] * clean["discount_pct"] / 100
clean["net_sales_inr"] = clean["gross_sales_inr"] - clean["discount_amount_inr"]

# Calendar features
clean["year"] = clean["order_date"].dt.year.astype("int64")
clean["month"] = clean["order_date"].dt.month.astype("int64")
clean["month_name"] = clean["order_date"].dt.month_name()
clean["year_month"] = clean["order_date"].dt.to_period("M").astype(str)

# Profit margin requires a cost field. The supplied schema has no cost/COGS field,
# so calculating a numeric profit margin would require an unsupported assumption.
# Keep a clearly documented placeholder rather than fabricating profitability.
clean["profit_margin_pct"] = np.nan

display(clean)

,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,gross_sales_inr,discount_amount_inr,net_sales_inr,year,month,month_name,year_month,profit_margin_pct
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid,1598,159.80,1438.20,2026,1,January,2026-01,NaN
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1,1499,0.0,Pending,1499,0.00,1499.00,2026,1,January,2026-01,NaN
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2397,119.85,2277.15,2026,1,January,2026-01,NaN
5,RT-1005,2026-01-09,Fresher,Unknown,Mentor Session,1,999,0.0,Failed,999,0.00,999.00,2026,1,January,2026-01,NaN
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,0.0,Paid,1998,0.00,1998.00,2026,1,January,2026-01,NaN
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,2,1499,0.0,Pending,2998,0.00,2998.00,2026,1,January,2026-01,NaN
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid,799,0.00,799.00,2026,1,January,2026-01,NaN
10,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15.0,Refunded,2998,449.70,2548.30,2026,1,January,2026-01,NaN


### Profit-margin limitation

A true profit margin is:

`(Net Sales - Cost) / Net Sales × 100`

The supplied dataset has `unit_price` but **no cost/COGS field**, so a defensible profit margin cannot be calculated. The notebook therefore records `profit_margin_pct` as missing rather than inventing a cost assumption. If a `cost_price` column is later supplied, the calculation can be added directly.

## 5. After cleaning — validation and proof

In [7]:
print("Final shape:", clean.shape)
print("\nFinal dtypes:")
display(clean.dtypes.to_frame("dtype"))

print("\nFinal missing values:")
display(clean.isna().sum().to_frame("missing_count"))

print("Duplicate rows:", clean.duplicated().sum())
print("Duplicate order IDs:", clean["order_id"].duplicated().sum())
print("Non-positive quantity:", (clean["quantity"] <= 0).sum())
print("Invalid discount:", ((clean["discount_pct"] < 0) | (clean["discount_pct"] > 100)).sum())

assert clean["order_id"].notna().all()
assert clean["order_id"].is_unique
assert clean["order_date"].notna().all()
assert clean["quantity"].gt(0).all()
assert clean["unit_price"].ge(0).all()
assert clean["discount_pct"].between(0, 100).all()

print("\nAll implemented business-rule assertions passed.")
display(clean)

Final shape: (8, 17)

Final dtypes:


,dtype
order_id,object
order_date,datetime64[ns]
customer_segment,string[python]
city,string[python]
category,string[python]
quantity,int64
unit_price,int64
discount_pct,float64
payment_status,string[python]
gross_sales_inr,int64



Final missing values:


,missing_count
order_id,0
order_date,0
customer_segment,0
city,0
category,0
quantity,0
unit_price,0
discount_pct,0
payment_status,0
gross_sales_inr,0


Duplicate rows: 0
Duplicate order IDs: 0
Non-positive quantity: 0
Invalid discount: 0

All implemented business-rule assertions passed.


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status,gross_sales_inr,discount_amount_inr,net_sales_inr,year,month,month_name,year_month,profit_margin_pct
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid,1598,159.80,1438.20,2026,1,January,2026-01,NaN
2,RT-1003,2026-01-05,Student,Chennai,Course Access,1,1499,0.0,Pending,1499,0.00,1499.00,2026,1,January,2026-01,NaN
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid,2397,119.85,2277.15,2026,1,January,2026-01,NaN
5,RT-1005,2026-01-09,Fresher,Unknown,Mentor Session,1,999,0.0,Failed,999,0.00,999.00,2026,1,January,2026-01,NaN
7,RT-1007,2026-01-12,Professional,Mumbai,Mentor Session,2,999,0.0,Paid,1998,0.00,1998.00,2026,1,January,2026-01,NaN
8,RT-1008,2026-01-14,Fresher,Bengaluru,Course Access,2,1499,0.0,Pending,2998,0.00,2998.00,2026,1,January,2026-01,NaN
9,RT-1009,2026-01-16,Student,Chennai,Learning Kit,1,799,0.0,Paid,799,0.00,799.00,2026,1,January,2026-01,NaN
10,RT-1010,2026-01-18,Professional,Delhi,Course Access,2,1499,15.0,Refunded,2998,449.70,2548.30,2026,1,January,2026-01,NaN


In [8]:
# Export clean standardized dataset
clean.to_csv(OUTPUT_FILE, index=False, date_format="%Y-%m-%d")
print(f"Saved: {OUTPUT_FILE.resolve()}")
print(f"Exported rows: {len(clean)}")
print(f"Exported columns: {len(clean.columns)}")

Saved: /mnt/data/clean_dataset.csv
Exported rows: 8
Exported columns: 17


## Final cleaning summary

| Issue | Action |
|---|---|
| Exact duplicate | Removed duplicate record |
| Mixed date formats / invalid date | Parsed with Pandas; rows with missing/invalid transaction dates removed because they cannot be inferred safely |
| Text quantity (`two`) | Converted to numeric `2` |
| Negative quantity | Removed because quantity must be > 0 |
| Missing discount | Filled with 0 under the data-dictionary rule |
| Discount > 100% | Treated as invalid and normalized to 0 |
| Missing city | Filled with explicit `Unknown` |
| Inconsistent category/status casing | Standardized capitalization |
| Date features | Added year, month, month name, and year-month |
| Sales features | Added gross sales, discount amount, and net sales |
| Profit margin | Not fabricated; requires cost/COGS, which is absent from the supplied schema |

**Deliverable:** `clean_dataset.csv`

**Data-volume caveat:** The provided raw file contains only 12 records. For a true 10,000+ row submission, replace the raw file with the larger source and rerun this notebook; the cleaning logic is row-count independent.